# Phase 4 — NLLB-200 embedding-swap architecture verification

**You run this notebook. Claude Code wrote it and cannot run it.**

Purpose: verify the single unverified assumption in
`src/kapampangan_morphbpe/nllb_contract.py`:

> "The future software stack must expose a genuinely independent source
> embedding without changing tied target embeddings or output projection."

NLLB-200-Distilled-600M ties one `shared` embedding (256,206 × 1,024) across
the **encoder input**, the **decoder input**, and the **output projection
(`lm_head`)**. The MorphBPE plan needs the *encoder* to get its own fresh
`nn.Embedding(6080, 1024)` — the only trainable parameters — while the
decoder and `lm_head` keep the original `shared` matrix, frozen and
untouched. This notebook does that surgery on the real weights and checks
every part of the assumption with a forward + backward pass.

### How to run
1. `Runtime → Change runtime type → T4 GPU` (free tier is plenty; CPU also
   works, ~2 min slower). No A100 needed — this is one forward/backward, not
   training.
2. `Runtime → Run all`. Total time ≈ 8–10 min (the 2.4 GB model download
   dominates).
3. The last cell writes and downloads
   `phase4-nllb-architecture-verification.json` / `.md`. Send the JSON back;
   it goes into `nllb/` in the repo.

Nothing here touches any repo file or uploads anything. It only downloads
public pretrained weights into this Colab runtime.

In [ ]:
%pip install -q -U "transformers>=4.44,<5" "sentencepiece" "sacremoses"
import json, math, platform, datetime
import torch
import transformers
print("transformers", transformers.__version__)
print("torch       ", torch.__version__)

In [ ]:
RESULTS = {
    "notebook": "phase4-nllb-architecture-verification",
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "purpose": (
        "verify the encoder source embedding of NLLB-200-Distilled-600M can be "
        "untied from the shared encoder/decoder/lm_head matrix without "
        "disturbing the target side"
    ),
    "versions": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
    },
}

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    RESULTS["runtime"] = {
        "device": "cuda",
        "gpu": prop.name,
        "vram_gib": round(prop.total_memory / 1024**3, 2),
    }
    DEVICE = "cuda"
else:
    RESULTS["runtime"] = {
        "device": "cpu",
        "note": "no GPU selected; a single 600M forward/backward still runs in ~1-2 min",
    }
    DEVICE = "cpu"

print(json.dumps(RESULTS["runtime"], indent=2))
print("Needs ~3 GB disk + ~6-10 GB RAM/VRAM. T4 (free) is enough.")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "facebook/nllb-200-distilled-600M"
print(f"downloading + loading {MODEL_NAME} (~2.4 GB) ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
model.eval()
cfg = model.config

RESULTS["model"] = {
    "name": MODEL_NAME,
    "class": type(model).__name__,
    "d_model": cfg.d_model,
    "vocab_size": cfg.vocab_size,
    "tie_word_embeddings": bool(getattr(cfg, "tie_word_embeddings", None)),
    "scale_embedding": bool(getattr(cfg, "scale_embedding", None)),
    "decoder_start_token_id": cfg.decoder_start_token_id,
    "pad_token_id": cfg.pad_token_id,
    "total_parameters": sum(p.numel() for p in model.parameters()),
}
print(json.dumps(RESULTS["model"], indent=2))

In [ ]:
import torch.nn as nn

# --- locate the tensors for the M2M100 (NLLB) layout, defensively ---
core = getattr(model, "model", None)
assert core is not None, f"unexpected top-level layout: {type(model).__name__} has no .model"
shared = core.shared
enc_embed = core.encoder.embed_tokens
dec_embed = core.decoder.embed_tokens
lm_head = model.lm_head


def same_tensor(a, b):
    return a.weight.data_ptr() == b.weight.data_ptr()


RESULTS["pre_surgery_tying"] = {
    "shared_shape": list(shared.weight.shape),
    "lm_head_shape": list(lm_head.weight.shape),
    "encoder_embed_is_shared": same_tensor(enc_embed, shared),
    "decoder_embed_is_shared": same_tensor(dec_embed, shared),
    "lm_head_is_shared": lm_head.weight.data_ptr() == shared.weight.data_ptr(),
    "get_input_embeddings_is_shared": (
        model.get_input_embeddings().weight.data_ptr() == shared.weight.data_ptr()
    ),
    "get_output_embeddings_is_lm_head": (
        model.get_output_embeddings().weight.data_ptr() == lm_head.weight.data_ptr()
    ),
}
print(json.dumps(RESULTS["pre_surgery_tying"], indent=2))
assert RESULTS["pre_surgery_tying"]["encoder_embed_is_shared"], (
    "expected the encoder embedding to start tied to shared; layout changed"
)

In [ ]:
# --- the surgery: give the ENCODER its own embedding, freeze everything else ---
SRC_VOCAB = 6080   # both Phase-5 conditions (morphbpe, penalty-8) are vocab 6,080
D_MODEL = cfg.d_model
SRC_PAD_ID = 0     # the MorphBPE artifacts use <pad> = 0 (see nllb_adapter.py)

for p in model.parameters():
    p.requires_grad_(False)

new_src_embed = nn.Embedding(SRC_VOCAB, D_MODEL, padding_idx=SRC_PAD_ID)
nn.init.normal_(new_src_embed.weight, mean=0.0, std=D_MODEL ** -0.5)
with torch.no_grad():
    new_src_embed.weight[SRC_PAD_ID].zero_()
new_src_embed.weight.requires_grad_(True)

# Direct attribute assignment on the encoder only. NOT set_input_embeddings(),
# which M2M100Model implements as "repoint shared AND encoder AND decoder".
core.encoder.embed_tokens = new_src_embed
model.to(DEVICE)

trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
RESULTS["surgery"] = {
    "source_vocab_size": SRC_VOCAB,
    "d_model": D_MODEL,
    "trainable_parameter_tensors": [n for n, _ in trainable],
    "trainable_parameter_count": sum(c for _, c in trainable),
    "expected_trainable_count": SRC_VOCAB * D_MODEL,
    "only_new_source_embedding_trains": (
        [n for n, _ in trainable] == ["model.encoder.embed_tokens.weight"]
    ),
}
print(json.dumps(RESULTS["surgery"], indent=2))

In [ ]:
# --- forward + backward, then check which tensors received a gradient ---
TGT_LANG = "tgl_Latn"  # NLLB-200 has Tagalog; there is no separate fil_Latn
try:
    if TGT_LANG not in tokenizer.additional_special_tokens:
        TGT_LANG = next(t for t in tokenizer.additional_special_tokens if t.endswith("_Latn"))
except Exception:
    pass
tokenizer.tgt_lang = TGT_LANG
RESULTS["target_language_code"] = TGT_LANG

torch.manual_seed(0)
B, S = 2, 7
src_ids = torch.randint(0, SRC_VOCAB, (B, S), device=DEVICE)
src_mask = torch.ones(B, S, dtype=torch.long, device=DEVICE)

lab = tokenizer(
    text_target=["Kumusta ka ngayon", "Salamat po sa tulong"],
    return_tensors="pt",
    padding=True,
)
labels = lab["input_ids"].to(DEVICE)
labels[labels == tokenizer.pad_token_id] = -100

model.train()  # frozen params stay frozen; just enables dropout paths
out = model(input_ids=src_ids, attention_mask=src_mask, labels=labels)
loss = out.loss
loss.backward()


def grad_sum(t):
    return None if t.grad is None else float(t.grad.detach().abs().sum().cpu())


fb = {
    "loss": float(loss.detach().cpu()),
    "logits_shape": list(out.logits.shape),
    "logits_last_dim_is_full_target_vocab": int(out.logits.shape[-1]) == int(cfg.vocab_size),
    "new_source_embedding_grad_sum": grad_sum(new_src_embed.weight),
    "shared_grad_sum": grad_sum(core.shared.weight),
    "decoder_embed_grad_sum": grad_sum(core.decoder.embed_tokens.weight),
    "lm_head_grad_sum": grad_sum(model.lm_head.weight),
}
fb["PASS_only_source_embedding_has_grad"] = bool(
    fb["new_source_embedding_grad_sum"] not in (None, 0.0)
    and fb["shared_grad_sum"] is None
    and fb["decoder_embed_grad_sum"] is None
    and fb["lm_head_grad_sum"] is None
)
fb["PASS_target_vocab_unchanged"] = bool(fb["logits_last_dim_is_full_target_vocab"])
RESULTS["forward_backward"] = fb
print(json.dumps(fb, indent=2))

In [ ]:
# --- does the target side stay tied + unchanged, even after an explicit re-tie? ---
after = {
    "encoder_embed_shape": list(core.encoder.embed_tokens.weight.shape),
    "encoder_embed_independent_of_shared": (
        core.encoder.embed_tokens.weight.data_ptr() != core.shared.weight.data_ptr()
    ),
    "decoder_embed_still_tied_to_shared": (
        core.decoder.embed_tokens.weight.data_ptr() == core.shared.weight.data_ptr()
    ),
    "lm_head_still_tied_to_shared": (
        model.lm_head.weight.data_ptr() == core.shared.weight.data_ptr()
    ),
    "shared_shape_unchanged": (
        list(core.shared.weight.shape) == [int(cfg.vocab_size), int(cfg.d_model)]
    ),
}
try:
    model.tie_weights()
    after["after_tie_weights_encoder_still_independent"] = bool(
        core.encoder.embed_tokens.weight.data_ptr() != core.shared.weight.data_ptr()
    )
    after["after_tie_weights_encoder_shape"] = list(core.encoder.embed_tokens.weight.shape)
    after["after_tie_weights_lm_head_still_tied"] = bool(
        model.lm_head.weight.data_ptr() == core.shared.weight.data_ptr()
    )
except Exception as exc:  # noqa: BLE001
    after["tie_weights_error"] = repr(exc)
RESULTS["post_surgery_tying"] = after
print(json.dumps(after, indent=2))

In [ ]:
# --- end-to-end plumbing: generation still runs (output text is meaningless) ---
model.eval()
try:
    with torch.no_grad():
        gen = model.generate(
            input_ids=src_ids,
            attention_mask=src_mask,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(TGT_LANG),
            max_new_tokens=12,
            num_beams=1,
        )
    RESULTS["generation_smoke"] = {
        "ran": True,
        "output_shape": list(gen.shape),
        "all_ids_in_target_vocab": bool((gen < int(cfg.vocab_size)).all().item()),
        "decoded_sample": tokenizer.batch_decode(gen, skip_special_tokens=True),
        "note": "encoder embedding is random -> decoded text is meaningless; this only "
                "confirms the encoder(new vocab) -> decoder(NLLB vocab) path executes",
    }
except Exception as exc:  # noqa: BLE001
    RESULTS["generation_smoke"] = {"ran": False, "error": repr(exc)}
print(json.dumps(RESULTS["generation_smoke"], indent=2))

In [ ]:
# --- verdict + write/download the report ---
g = RESULTS
checks = {
    "encoder_starts_tied_to_shared": g["pre_surgery_tying"]["encoder_embed_is_shared"],
    "encoder_embedding_swapped_independently": g["post_surgery_tying"][
        "encoder_embed_independent_of_shared"
    ],
    "decoder_and_lm_head_stay_tied_and_shape_unchanged": bool(
        g["post_surgery_tying"]["decoder_embed_still_tied_to_shared"]
        and g["post_surgery_tying"]["lm_head_still_tied_to_shared"]
        and g["post_surgery_tying"]["shared_shape_unchanged"]
    ),
    "only_new_source_embedding_is_trainable": bool(
        g["surgery"]["only_new_source_embedding_trains"]
        and g["surgery"]["trainable_parameter_count"] == g["surgery"]["expected_trainable_count"]
    ),
    "backward_updates_only_source_embedding": g["forward_backward"][
        "PASS_only_source_embedding_has_grad"
    ],
    "target_output_vocab_unchanged": g["forward_backward"]["PASS_target_vocab_unchanged"],
    "explicit_retie_keeps_encoder_independent": g["post_surgery_tying"].get(
        "after_tie_weights_encoder_still_independent", False
    ),
    "generation_runs_end_to_end": g.get("generation_smoke", {}).get("ran", False),
}
g["checks"] = checks
g["VERDICT"] = (
    "PASS - embedding-swap assumption holds"
    if all(checks.values())
    else "FAIL / PARTIAL - inspect the checks below and the full JSON"
)
print(json.dumps({"checks": checks, "VERDICT": g["VERDICT"]}, indent=2))

json_path = "phase4-nllb-architecture-verification.json"
md_path = "phase4-nllb-architecture-verification.md"
with open(json_path, "w") as fh:
    json.dump(g, fh, indent=2)

lines = [
    "# Phase 4 - NLLB-200 architecture verification",
    "",
    f"**{g['VERDICT']}**",
    "",
    f"- model: {g['model']['name']} ({g['model']['class']})",
    f"- d_model {g['model']['d_model']}, target vocab {g['model']['vocab_size']}, "
    f"tie_word_embeddings={g['model']['tie_word_embeddings']}, "
    f"scale_embedding={g['model']['scale_embedding']}",
    f"- source vocab tested: {g['surgery']['source_vocab_size']}",
    f"- trainable params after surgery: {g['surgery']['trainable_parameter_count']:,} "
    f"(expected {g['surgery']['expected_trainable_count']:,})",
    f"- forward loss: {g['forward_backward']['loss']:.4f}",
    f"- target language code: {g.get('target_language_code')}",
    "",
    "## checks",
    "",
]
for name, ok in checks.items():
    lines.append(f"- [{'x' if ok else ' '}] {name}")
lines.append("")
with open(md_path, "w") as fh:
    fh.write("\n".join(lines) + "\n")

print(f"\nwrote {json_path} and {md_path}")
try:
    from google.colab import files

    files.download(json_path)
    files.download(md_path)
except Exception as exc:  # noqa: BLE001
    print("auto-download unavailable:", exc)
    print("Download both files from the Colab file browser (left panel).")

## Interpreting the result

**All checks pass →** the source-embedding-swap approach in
`nllb_contract.py` / `nllb_adapter.py` is architecturally sound: the encoder
can take a small independent MorphBPE embedding as the only trainable
parameters, the Filipino/Tagalog decoder + output projection are provably
untouched, and gradients flow only where intended. Phase 5 can proceed on
this design.

**Any check fails →** the notebook's JSON records exactly which tensor
identity / gradient / shape expectation broke. Common causes and fixes:
- `explicit_retie_keeps_encoder_independent` is false → `from_pretrained`
  or `tie_weights()` re-points the encoder; Phase 5 must re-apply the swap
  *after* any such call, or set `config.tie_word_embeddings=False` on the
  encoder side only.
- `backward_updates_only_source_embedding` is false with a non-None
  `shared_grad_sum` → the frozen `shared` still receives gradient through
  the decoder/lm_head path; Phase 5's optimizer must filter on
  `requires_grad`, which this notebook already sets correctly, so this would
  indicate a deeper tying issue to escalate.

Send `phase4-nllb-architecture-verification.json` back to Claude Code — it
gets committed under `nllb/` and the "unverified integration property" note
in `nllb_contract.py` gets updated with the verdict.